# 06 Author Style Embeddings

Goal: load a trained passage embedder, embed multiple passages per author, attention-pool them into one corpus-level style embedding per author, and save the result for the Style LLM.

In [ ]:
import os
import sys
from collections import defaultdict
from pathlib import Path

def find_project_root():
    env_root = os.getenv("VOICE_CLONE_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "voice_clone").exists():
            return candidate
    raise RuntimeError("Could not find project root. Set VOICE_CLONE_ROOT to the repo path.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoTokenizer

from voice_clone.data.pipeline import load_passages
from voice_clone.models import CorpusStylePooler, StyleEmbedder

print("ROOT:", ROOT)

In [ ]:
PASSAGES_PATH = ROOT / "data" / "processed" / "passages.jsonl"
CHECKPOINT_PATH = ROOT / "checkpoints" / "style_embedder_stage3_best.pt"
OUTPUT_PATH = ROOT / "data" / "processed" / "author_style_embeddings.pt"

MODEL_NAME = "allenai/longformer-base-4096"
MAX_LENGTH = 768
PASSAGES_PER_AUTHOR = 12
EMBED_BATCH_SIZE = 2

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)
print("checkpoint:", CHECKPOINT_PATH)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
embedder = StyleEmbedder.from_pretrained(MODEL_NAME).to(device)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
embedder.load_state_dict(checkpoint["model_state_dict"])
embedder.eval()

# This pooler is intentionally separate and lightweight. Later we can train it;
# for now it gives us attention-weighted corpus embeddings instead of averages.
corpus_pooler = CorpusStylePooler(embedding_dim=512).to(device)
corpus_pooler.eval()

print("loaded checkpoint step:", checkpoint.get("step"))
print("checkpoint metrics:", checkpoint.get("metrics"))

In [ ]:
passages = load_passages(PASSAGES_PATH)
by_author = defaultdict(list)
for passage in passages:
    by_author[passage.author_id].append(passage)

selected = {
    author: rows[:PASSAGES_PER_AUTHOR]
    for author, rows in sorted(by_author.items())
    if rows
}

print({author: len(rows) for author, rows in selected.items()})

In [ ]:
def encode_texts(texts):
    encoded = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    return {key: value.to(device) for key, value in encoded.items()}


def embed_texts(texts):
    outputs = []
    with torch.no_grad():
        for start in range(0, len(texts), EMBED_BATCH_SIZE):
            encoded = encode_texts(texts[start:start + EMBED_BATCH_SIZE])
            outputs.append(embedder(**encoded).cpu())
    return torch.cat(outputs, dim=0)

In [ ]:
author_embeddings = {}
passage_embedding_counts = {}

for author, rows in selected.items():
    print(f"embedding author={author} passages={len(rows)}")
    texts = [row.text for row in rows]
    passage_embeddings = embed_texts(texts).to(device)
    with torch.no_grad():
        pooled = corpus_pooler(passage_embeddings.unsqueeze(0)).squeeze(0).cpu()
    author_embeddings[author] = pooled
    passage_embedding_counts[author] = len(rows)

stacked = torch.stack(list(author_embeddings.values()))
print("authors:", len(author_embeddings))
print("embedding shape:", tuple(stacked.shape))
print("norms:", stacked.norm(dim=-1).round(decimals=4).tolist())

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "author_embeddings": author_embeddings,
        "passage_counts": passage_embedding_counts,
        "model_name": MODEL_NAME,
        "checkpoint_path": str(CHECKPOINT_PATH),
        "passages_per_author": PASSAGES_PER_AUTHOR,
        "embedding_dim": 512,
    },
    OUTPUT_PATH,
)
print("saved:", OUTPUT_PATH)